# Stage 3: Major Axis Detection (Downsampled)

Compute major axis using PCA and add as visualization layer.

**Input**: `labelsShrunk_50_ds/*_shrunk.nii.gz`  
**Output**: `labelsWithAxis_ds/*_with_axis.nii.gz`  
**Metrics**: `metrics/axis_info_ds.json`

In [1]:
# Configuration
TARGET_DIR = "/mnt/c/users/mwild/firebase/perios/levi_data_1.6.26"

# Axis visualization parameters
AXIS_THICKNESS = 3
AXIS_VALUE = 2  # Class value for axis voxels

In [2]:
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    load_nifti, save_nifti, save_metrics, ensure_dir,
    compute_major_axis, draw_line_3d
)

In [3]:
# Setup directories (using downsampled data)
target = Path(TARGET_DIR)
INPUT_DIR = target / "labelsShrunk_50_ds"
OUTPUT_DIR = ensure_dir(target / "labelsWithAxis_ds")
METRICS_DIR = ensure_dir(target / "metrics")

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

Input:  /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsShrunk_50_ds
Output: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsWithAxis_ds


In [4]:
def add_axis_to_mask(mask_data, thickness=3, axis_value=2):
    """Add major axis visualization to mask."""
    # Compute major axis
    axis_info = compute_major_axis(mask_data)
    
    # Create output with mask as class 1
    output = (mask_data > 0).astype(np.uint8)
    
    # Draw axis line
    start, end = axis_info['endpoints']
    axis_line = draw_line_3d(mask_data.shape, start, end, thickness=thickness, value=axis_value)
    
    # Add axis to output (axis overrides mask where they overlap)
    output = np.where(axis_line > 0, axis_value, output)
    
    return output, axis_info


def process_axis(input_path, output_path):
    """Process a single mask to add major axis."""
    # Load mask
    data, affine, header = load_nifti(input_path)
    mask_data = (data > 0).astype(np.uint8)
    
    # Add major axis
    output_data, axis_info = add_axis_to_mask(mask_data, 
                                               thickness=AXIS_THICKNESS, 
                                               axis_value=AXIS_VALUE)
    
    # Save output
    save_nifti(output_data, affine, header, output_path)
    
    return axis_info

In [5]:
# Process all shrunk masks
mask_files = sorted(INPUT_DIR.glob("*_shrunk.nii.gz"))
print(f"Found {len(mask_files)} shrunk masks to process")
print(f"Axis will be drawn with thickness={AXIS_THICKNESS}, class={AXIS_VALUE}")
print("="*70)

all_axis_info = []

for idx, mask_file in enumerate(mask_files, 1):
    sample_name = mask_file.stem.replace('_shrunk', '').replace('.nii', '')
    output_file = OUTPUT_DIR / f"{sample_name}_with_axis.nii.gz"
    
    print(f"[{idx}/{len(mask_files)}] {sample_name}")
    
    try:
        axis_info = process_axis(mask_file, output_file)
        axis_info['filename'] = mask_file.name
        axis_info['sample_name'] = sample_name
        all_axis_info.append(axis_info)
        
        print(f"    Length: {axis_info['length']:.1f} voxels")
        print(f"    Variance explained: {axis_info['explained_variance']:.1%}")
    except Exception as e:
        print(f"    ERROR: {e}")

Found 30 shrunk masks to process
Axis will be drawn with thickness=3, class=2
[1/30] Digit105


    Length: 175.0 voxels
    Variance explained: 76.1%
[2/30] Digit10


    Length: 189.2 voxels
    Variance explained: 75.3%
[3/30] Digit12


    Length: 194.3 voxels
    Variance explained: 75.5%
[4/30] Digit23


    Length: 196.9 voxels
    Variance explained: 79.0%
[5/30] Digit26


...

    Length: 134.2 voxels
    Variance explained: 64.6%
[27/30] Digit95


    Length: 214.1 voxels
    Variance explained: 78.5%
[28/30] Digit96


    Length: 134.7 voxels
    Variance explained: 64.9%
[29/30] Digit97


    Length: 211.0 voxels
    Variance explained: 79.3%
[30/30] Digit99


    Length: 148.5 voxels
    Variance explained: 67.7%


In [6]:
# Save metrics
metrics_file = METRICS_DIR / "axis_info_ds.json"
save_metrics(all_axis_info, metrics_file)

# Summary
print("\n" + "="*70)
print("AXIS GENERATION COMPLETE (DOWNSAMPLED)")
print("="*70)
print(f"\nProcessed: {len(all_axis_info)}/{len(mask_files)} masks")

if all_axis_info:
    avg_length = np.mean([a['length'] for a in all_axis_info])
    avg_variance = np.mean([a['explained_variance'] for a in all_axis_info])
    print(f"Average axis length: {avg_length:.1f} voxels (downsampled)")
    print(f"Average variance explained: {avg_variance:.1%}")

print(f"\nOutput: {OUTPUT_DIR}")
print(f"Metrics: {metrics_file}")
print(f"\nClasses: 0=background, 1=mask, {AXIS_VALUE}=axis")


AXIS GENERATION COMPLETE (DOWNSAMPLED)

Processed: 30/30 masks
Average axis length: 182.2 voxels (downsampled)
Average variance explained: 74.6%

Output: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/labelsWithAxis_ds
Metrics: /mnt/c/users/mwild/firebase/perios/levi_data_1.6.26/metrics/axis_info_ds.json

Classes: 0=background, 1=mask, 2=axis
